In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession


In [2]:
FATOS = [
    "reproducoes",
    "pagamentos",
    "assinaturas",
]


In [3]:
load_dotenv()

is_docker = os.path.exists('/.dockerenv')

POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost") if is_docker else "localhost"
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")
POSTGRES_DB   = os.getenv("POSTGRES_DB",   "streaming")
POSTGRES_USER = os.getenv("POSTGRES_USER", "streaming")
POSTGRES_PASS = os.getenv("POSTGRES_PASSWORD", "streaming123")

MINIO_HOST = "minio:9000" if is_docker else "localhost:9000"
MINIO_USER = os.getenv("MINIO_ROOT_USER",     "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

spark = (
    SparkSession.builder
    .appName("landing-fatos")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .getOrCreate()
)

cliente_minio = Minio(
    MINIO_HOST,
    access_key=MINIO_USER,
    secret_key=MINIO_PASS,
    secure=False,
)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/16 15:19:39 WARN Utils: Your hostname, DESKTOP-84BB517, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/16 15:19:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/mnt/c/Users/Laura/Documents/Disciplinas_SATC/engenharia_dados/projeto-ed-final/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/lauras/.ivy2.5.2/cache
The jars for the packages stored in: /home/lauras/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-9ecdb754-6a7f-4fe7-b88e-2cf4bed30d8d;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
downloading https://repo1.maven.org/maven2/org/pos

In [4]:
buckets = {b.name for b in cliente_minio.list_buckets()}
print("Buckets disponíveis:", buckets)

if "landing" not in buckets:
    cliente_minio.make_bucket("landing")
    print("Bucket 'landing' criado.")


Buckets disponíveis: set()
Bucket 'landing' criado.


In [5]:
jdbc_url = f"jdbc:postgresql://{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"

propriedades = {
    "user":   POSTGRES_USER,
    "password": POSTGRES_PASS,
    "driver": "org.postgresql.Driver",
}

for tabela in FATOS:
    print(f"\n{'=' * 50}")
    print(f"Processando: {tabela}")
    print(f"{'=' * 50}")

    df = spark.read.jdbc(url=jdbc_url, table=tabela, properties=propriedades)
    total = df.count()
    print(f"Registros lidos: {total:,}")

    caminho_local = f"/tmp/landing/{tabela}"

    df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(caminho_local)

    csv_dir = Path(caminho_local)
    arquivo_csv = next(
        f for f in csv_dir.iterdir() if f.suffix == ".csv"
    )

    object_name = f"{tabela}/{tabela}.csv"
    cliente_minio.fput_object("landing", object_name, str(arquivo_csv))
    print(f"Enviado → landing/{object_name}")

print("\nIngestão das tabelas fato para a Landing concluída.")



Processando: reproducoes
Registros lidos: 70,000
Enviado → landing/reproducoes/reproducoes.csv

Processando: pagamentos
Registros lidos: 10,500
Enviado → landing/pagamentos/pagamentos.csv

Processando: assinaturas
Registros lidos: 3,500
Enviado → landing/assinaturas/assinaturas.csv

Ingestão das tabelas fato para a Landing concluída.


In [6]:
spark.stop()
